In [1]:
##Primero importamos pandas y numpy
import pandas as pd
import numpy as np
import re

In [2]:
#pip install xlrd

In [3]:
##Descargamos y miramos el archivo
df = pd.read_excel("GSAF5.xls")
df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,10th January,2026.0,Unprovoked,Australia,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,...,Unknown,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8th January,2026.0,Unprovoked,US Virgin Islands,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,...,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3rd January,2026.0,Unprovoked,New Caledonia,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,...,Unknown,Andy Currie: Province Sud:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,...,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,...,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
##Miramos el tamaño de la tabla e info general 
df.shape

(7065, 23)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7065 entries, 0 to 7064
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            7065 non-null   object 
 1   Year            7063 non-null   float64
 2   Type            7047 non-null   object 
 3   Country         7015 non-null   object 
 4   State           6578 non-null   object 
 5   Location        6498 non-null   object 
 6   Activity        6480 non-null   object 
 7   Name            6846 non-null   object 
 8   Sex             6486 non-null   object 
 9   Age             4070 non-null   object 
 10  Injury          7030 non-null   object 
 11  Fatal Y/N       6504 non-null   object 
 12  Time            3538 non-null   object 
 13  Species         3934 non-null   object 
 14  Source          7045 non-null   object 
 15  pdf             6799 non-null   object 
 16  href formula    6794 non-null   object 
 17  href            6796 non-null   o

### Cambiar los títulos a minúsculas y quitar espacios

In [6]:
df.columns = df.columns.str.lower().str.replace(" ","")
df.head()

,date,year,type,country,state,location,activity,name,sex,age,...,species,source,pdf,hrefformula,href,casenumber,casenumber.1,originalorder,unnamed:21,unnamed:22
0,10th January,2026.0,Unprovoked,Australia,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,...,Unknown,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8th January,2026.0,Unprovoked,US Virgin Islands,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,...,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3rd January,2026.0,Unprovoked,New Caledonia,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,...,Unknown,Andy Currie: Province Sud:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,...,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,...,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Eliminamos las columnas sin datos

In [7]:
columns_to_drop = { 
    'pdf',
    'href',
    'hrefformula',
    'casenumber',
    'casenumber.1',
    'originalorder',
    'unnamed:21',
    'unnamed:22'
}
df = df.drop(columns=columns_to_drop)

### Eliminar duplicados

In [8]:
##Miramos los duplicados
df.duplicated().sum()

np.int64(1)

In [9]:
##Buscamos el único duplicado que existe, sobre todo porque es una muestra muy pequeña 
filas_duplicadas = df[df.duplicated(keep=False)]
filas_duplicadas

,date,year,type,country,state,location,activity,name,sex,age,injury,fataly/n,time,species,source
5436,Fall 1943,1943.0,Unprovoked,USA,Hawaii,"Midway Island, Northwestern Hawaiian Islands",Spearfishing,2 males,M,NaN,Calf nipped in each case,N,NaN,"""small sharks""",W. M. Chapman
5437,Fall 1943,1943.0,Unprovoked,USA,Hawaii,"Midway Island, Northwestern Hawaiian Islands",Spearfishing,2 males,M,NaN,Calf nipped in each case,N,NaN,"""small sharks""",W. M. Chapman


In [10]:
##Al verla, decidimoso eliminar la primera fila
df = df.drop(5436)

### Gestionamos los nulos

In [11]:
##Comprobamos cuántos nulos hay
df.isnull().sum()

date           0
year           2
type          18
country       50
state        487
location     567
activity     585
name         219
sex          579
age         2994
injury        35
fataly/n     561
time        3526
species     3131
source        20
dtype: int64

In [12]:
##Comprobamos el porcentaje de los datos para tomar las decisiones respecto a nuestras hipótesis
df.isnull().sum()/df.shape[0]

date        0.000000
year        0.000283
type        0.002548
country     0.007078
state       0.068941
location    0.080266
activity    0.082814
name        0.031002
sex         0.081965
age         0.423839
injury      0.004955
fataly/n    0.079417
time        0.499151
species     0.443233
source      0.002831
dtype: float64

In [13]:
## Decidimos en qué columnas vamos a trabajar basándonos en nuestras hipótesis: 
## 'Sex', 'Year', 'Type', 'Fatal', 'Date' y 'Country'
## Comenzamos a investigar qué tipos de datos nulos son y tomar decisiones: eliminar filas o imputar valores. 

### HIPÓTESIS 1 - 'SEX': ¿Mueren más hombres que mujeres?

In [14]:
##Creamos nuestro df específico para esta columna
df_sex = df[['sex']]

In [15]:
## Comprobamos que 'Sex' tiene valores nulos
df_sex.isnull().sum()/df.shape[0]

sex    0.081965
dtype: float64

In [16]:
##Eliminamos los valores nulos que tiene
df_sex = df_sex.dropna(subset=['sex'])

In [17]:
##Comprobamos cómo son los datos que tenemos, es decir, qué valores se han asociado a la serie. 
df_sex['sex'].unique()

array(['M', 'F', 'F ', 'M ', ' M', 'm', 'lli', 'M x 2', 'N', '.'],
      dtype=object)

In [18]:
## Nos damos cuenta de que hay valores que aunque no son nulos, no son válidos para hacer el análisis.
df_sex['sex'] = df_sex['sex'].str.strip().str.upper()

In [19]:
### Como solo hay cinco valores dierentes de 'M' y 'F' decidimos prescindir de ellos
df_sex['sex'].value_counts()

sex
M        5670
F         810
N           2
LLI         1
M X 2       1
.           1
Name: count, dtype: int64

In [20]:
## Vamos a filtrar todos los valores correctos
filtered_values = ['F', 'M']
df_sex = df_sex[df_sex['sex'].isin(filtered_values)]

In [21]:
df_sex['sex'].unique()

array(['M', 'F'], dtype=object)

In [22]:
## Con los datos ya limpios, aplicamos la función para responder a nuestra hipótesis
df_sex.describe()

,sex
count,6480
unique,2
top,M
freq,5670


- Nuestros resultados indican que del total de registros (6480) la mayoría de muertes son de hombres (M) con una frecuencia de 5670. Eso es el 87,5% de hombres frente al total de 810 mujeres, es decir, 12,5% del total. 

### HIPÓTESIS 2 - 'YEAR': ¿En qué año hubo más ataques?

In [23]:
## Repetimos el mismo proceso que hicimos con 'sex'
##Creamos nuestro df específico para esta columna
df_year = df[['year']]

In [24]:
## Comprobamos que 'year' NO tiene valores nulos y que está en float. 
df_year.isnull().sum()/df.shape[0]

year    0.000283
dtype: float64

In [25]:
##Comprobamos cómo son los datos que tenemos, es decir, qué valores se han asociado a la serie. 
df_year['year'].unique()

array([2026., 2025., 2024., 2023., 2022., 2021., 2020., 2019., 2018.,
       2017.,   nan, 2016., 2015., 2014., 2013., 2012., 2011., 2010.,
       2009., 2008., 2007., 2006., 2005., 2004., 2003., 2002., 2001.,
       2000., 1999., 1998., 1997., 1996., 1995., 1984., 1994., 1993.,
       1992., 1991., 1990., 1989., 1969., 1988., 1987., 1986., 1985.,
       1983., 1982., 1981., 1980., 1979., 1978., 1977., 1976., 1975.,
       1974., 1973., 1972., 1971., 1970., 1968., 1967., 1966., 1965.,
       1964., 1963., 1962., 1961., 1960., 1959., 1958., 1957., 1956.,
       1955., 1954., 1953., 1952., 1951., 1950., 1949., 1948., 1848.,
       1947., 1946., 1945., 1944., 1943., 1942., 1941., 1940., 1939.,
       1938., 1937., 1936., 1935., 1934., 1933., 1932., 1931., 1930.,
       1929., 1928., 1927., 1926., 1925., 1924., 1923., 1922., 1921.,
       1920., 1919., 1918., 1917., 1916., 1915., 1914., 1913., 1912.,
       1911., 1910., 1909., 1908., 1907., 1906., 1905., 1904., 1903.,
       1902., 1901.,

In [26]:
## Hacemos un análisis de los datos de year y decidimos quedarnos con los valores mayores de 1000. 
df_year['year'] = pd.to_numeric(df_year['year'], errors='coerce')
df_year.loc[df_year['year'] <= 1000, 'year'] = np.nan

/var/folders/bk/hzstscb16rn4p883jsg8t31c0000gn/T/ipykernel_4582/3964959132.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_year['year'] = pd.to_numeric(df_year['year'], errors='coerce')
/var/folders/bk/hzstscb16rn4p883jsg8t31c0000gn/T/ipykernel_4582/3964959132.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_year.loc[df_year['year'] <= 1000, 'year'] = np.nan


In [27]:
## Para saber cuántos nulos hay respecto a filas originales decidimos hacer dos prints para tenerlo más claro
print("Valores nulos totales:", df_year['year'].isna().sum())
print("Filas originales:", df_year.shape[0])

Valores nulos totales: 134
Filas originales: 7064


In [28]:
## No habiendo nulos, descubrimos que tenemos 6371 valores válidos.
## Respondemos a nuestra hipótesis: ¿En qué año hubo más muertes? 
## Aunque podemos calcular la moda para descubrirlo, preferimos tener el ranking total.
df_year.value_counts()

year  
2015.0    143
2017.0    141
2016.0    133
2011.0    128
2014.0    126
         ... 
1787.0      1
1786.0      1
1785.0      1
1784.0      1
1500.0      1
Name: count, Length: 257, dtype: int64

- Nuestros datos indican que el año en el que ha muerto más gente ha sido el 2015 con 130 registros de 6371. En segunda posición, con solo dos muertes menos, fue 2017 y en tercer lugar, con 122 registros, el año 2016.

### HIPÓTESIS 3 - 'TYPE' y 'FATAL': ¿Cuál es el ratio entre provocado y muerte?

In [29]:
### Hacemos un df más pequeño solo con las series que queremos
df_hip3 = df[['name', 'type', 'fataly/n']]

In [30]:
### Miramos los valores que se han asociado a la columna type de nnuestro df pequeño
df_hip3['type'].unique()

array(['Unprovoked', 'Provoked', 'Questionable', 'unprovoked',
       ' Provoked', 'Watercraft', 'Sea Disaster', nan, '?', 'Unconfirmed',
       'Unverified', 'Invalid', 'Under investigation', 'Boat'],
      dtype=object)

In [31]:
### Eliminamos espacios y reemplazamos los valores que no queremos a las categorías que preferimos
df_hip3['type'] = df_hip3['type'].str.strip()
df_hip3['type'] = df_hip3['type'].replace({
    'unprovoked': 'Unprovoked',
    'Unverified': 'Unknown',
    'Unconfirmed': 'Unknown',
    '?': 'Unknown',
    'Nan': 'Unknown',         
    'Invalid': 'Unknown',
    'Under investigation': 'Unknown',
    'Questionable' : 'Unknown',
    'Watercraft': 'Other', 
    'Sea disaster': 'Other',
    'Sea Disaster': 'Other', 
    'Boat': 'Other'
})

/var/folders/bk/hzstscb16rn4p883jsg8t31c0000gn/T/ipykernel_4582/3796310660.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hip3['type'] = df_hip3['type'].str.strip()
/var/folders/bk/hzstscb16rn4p883jsg8t31c0000gn/T/ipykernel_4582/3796310660.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hip3['type'] = df_hip3['type'].replace({


In [32]:
### Comprbamos que están unificados
df_hip3['type'].unique()

array(['Unprovoked', 'Provoked', 'Unknown', 'Other', nan], dtype=object)

In [33]:
### Eliminamos valores nulos
df_hip3 = df_hip3.dropna(subset='type')

In [34]:
### Comprobamos que no quedan valores nulos
df_hip3.isnull().sum()

name        219
type          0
fataly/n    557
dtype: int64

- 'FATAL'

In [35]:
### Miramos valores únicos de la serie 'Fatal'
df_hip3['fataly/n'].unique()

array(['N', 'Y', 'F', 'M', nan, 'n', 'Nq', 'UNKNOWN', 2017, 'Y x 2', ' N',
       'N ', 'y'], dtype=object)

In [36]:
### Quitamos espacios en blanco y ponemos todos en mayúscula, para unificar valores
df_hip3['fataly/n'] = df_hip3['fataly/n'].str.strip().str.upper()
df_hip3['fataly/n'].unique()

array(['N', 'Y', 'F', 'M', nan, 'NQ', 'UNKNOWN', 'Y X 2'], dtype=object)

In [37]:
### Creamos filtro para que solo guarde Yes y No como valores
filtered_values = ['Y', 'N']
df_hip3 = df_hip3[df_hip3['fataly/n'].isin(filtered_values)]


In [38]:
### Comprobamos que ha funcionado 
df_hip3['fataly/n'].unique()

array(['N', 'Y'], dtype=object)

In [39]:
### Comprobamos que ya no quedan nulos
df_hip3.isnull().sum()

name        151
type          0
fataly/n      0
dtype: int64

In [40]:
### Respondemos a la hipótesis: ¿Cuál es el ratio de provocados y muertes?
### Hacemos un conteo del total de personas en nuestro df

In [41]:
df_hip3['name'].count()

np.int64(6257)

In [42]:
### Asociamos variables según nuestras hipótesis para averiguar el total de personas que cumplan las condiciones

In [43]:
dead_provoked = df_hip3.loc[(df_hip3['type'] == 'Provoked') & (df_hip3['fataly/n'] == 'Y')]

In [44]:
dead_provoked.count()

name        21
type        21
fataly/n    21
dtype: int64

In [45]:
notdead_provoked = df_hip3.loc[(df_hip3['type'] == 'Provoked') & (df_hip3['fataly/n'] == 'N')]

In [46]:
notdead_provoked.count()

name        605
type        613
fataly/n    613
dtype: int64

In [47]:
dead_notprovoked = df_hip3.loc[(df_hip3['type'] == 'Unprovoked') & (df_hip3['fataly/n'] == 'Y')]

In [48]:
dead_notprovoked.count()

name        1235
type        1266
fataly/n    1266
dtype: int64

In [49]:
notdead_unprovoked = df_hip3.loc[(df_hip3['type'] == 'Unprovoked') & (df_hip3['fataly/n'] == 'N')]

In [50]:
notdead_unprovoked.count()

name        3827
type        3872
fataly/n    3872
dtype: int64

In [51]:
unknown_causes = df_hip3.loc[(df_hip3['type'] == 'Unknown')]

In [52]:
unknown_causes.count()

name        49
type        50
fataly/n    50
dtype: int64

In [53]:
other_causes = df_hip3.loc[(df_hip3['type'] == 'Other')]

In [54]:
other_causes.count()

name        520
type        586
fataly/n    586
dtype: int64

- Descubrimos que 21 personas fallecieron por haber provocado el ataque y 613 que lo provocaron sobrevivieron. Por otro lado, murieron 1266 personas aun no provocando el ataque y atacaron a 3872 personas sin provocarlo. Hay 50 personas atacadas (incluyendo fallecidas y vivas) en las que no se sabe si se provocó o no al tiburón y 586 personas fueron atacadas (fallecidas y vivas) por los tiburones con otra causa (desastres naturales, supervivencia etc.)

### HIPÓTESIS 4 - 'COUNTRY': ¿En qué lugar ha habido más ataques?

In [55]:
### Hacemos un df más pequeño para utilizar country
df_country = df[['country']]

In [56]:
### Comprobamos valores únicos 
df_country['country'].unique()

array(['Australia', 'US Virgin Islands', 'New Caledonia', 'USA',
       'French Polynesia', 'Samoa', 'Columbia', 'Costa Rica', 'Bahamas',
       'Puerto Rico', 'Spain', 'Canary Islands', 'South Africa',
       'Vanuatu', 'Jamaica', 'Israel', 'Mexico', 'Maldives',
       'Philippines', 'Turks and Caicos', 'Mozambique', 'Egypt',
       'Thailand', 'New Zealand', 'Hawaii', 'Honduras', 'Indonesia',
       'Morocco', 'Belize', 'Maldive Islands', 'Tobago', 'AUSTRALIA',
       'INDIA', 'TRINIDAD', 'BAHAMAS', 'SOUTH AFRICA', 'MEXICO',
       'NEW ZEALAND', 'EGYPT', 'BELIZE', 'PHILIPPINES', 'Coral Sea',
       'SPAIN', 'PORTUGAL', 'SAMOA', 'COLOMBIA', 'ECUADOR',
       'FRENCH POLYNESIA', 'NEW CALEDONIA', 'TURKS and CaICOS', 'CUBA',
       'BRAZIL', 'SEYCHELLES', 'ARGENTINA', 'FIJI', 'MeXICO', 'ENGLAND',
       'JAPAN', 'INDONESIA', 'JAMAICA', 'MALDIVES', 'THAILAND',
       'COLUMBIA', 'COSTA RICA', 'British Overseas Territory', 'CANADA',
       'JORDAN', 'ST KITTS / NEVIS', 'ST MARTIN', 'PAPUA

In [57]:
### Unificamos para que todo sea mayúscula y quitando espacios en blanco
df_country['country'] = df_country['country'].str.upper().str.strip()

/var/folders/bk/hzstscb16rn4p883jsg8t31c0000gn/T/ipykernel_4582/1286141579.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_country['country'] = df_country['country'].str.upper().str.strip()


In [58]:
### Observamos que se unifican algunos valores 
df_country['country'].value_counts()

country
USA                   2577
AUSTRALIA             1516
SOUTH AFRICA           599
NEW ZEALAND            146
BAHAMAS                141
                      ... 
RED SEA                  1
BRITISH ISLES            1
WESTERN SAMOA            1
SOUTH CHINA SEA          1
CEYLON (SRI LANKA)       1
Name: count, Length: 213, dtype: int64

In [59]:
### Lo pasamos a series para observar con cuántos valores unicos estamos tratando 
pd.Series(df_country['country'].unique())

0               AUSTRALIA
1       US VIRGIN ISLANDS
2           NEW CALEDONIA
3                     USA
4        FRENCH POLYNESIA
              ...        
209               BAHREIN
210                 KOREA
211              RED SEA?
212                 ASIA?
213    CEYLON (SRI LANKA)
Length: 214, dtype: object

In [60]:
### Nos quedamos con el primer país que aparece en los registros 
df_country['country'] = df_country['country'].str.split('/').str[0].str.strip()

/var/folders/bk/hzstscb16rn4p883jsg8t31c0000gn/T/ipykernel_4582/2648831502.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_country['country'] = df_country['country'].str.split('/').str[0].str.strip()


In [61]:
### Creamos un diccionario para agrupar los países que son el mismo aunque aparezcan escritos de manera diferente
other= {
    'ST KITTS / NEVIS': 'SAINT KITTS AND NEVIS',
    'ST KITTS': 'SAINT KITTS AND NEVIS',
    'NEVIS': 'SAINT KITTS AND NEVIS',
    'TRINIDAD & TOBAGO': 'TRINIDAD AND TOBAGO',
    'TRINIDAD': 'TRINIDAD AND TOBAGO',
    'TURKS & CAICOS': 'TURKS AND CAICOS',
    'ST. MARTIN': 'SAINT MARTIN',
    'ST. MAARTIN': 'SAINT MARTIN',
    'MALDIVE ISLANDS': 'MALDIVES',
    'COLUMBIA': 'COLOMBIA',

    'BRITISH VIRGIN ISLANDS': 'UNITED KINGDOM',
    'ST HELENA, BRITISH OVERSEAS TERRITORY': 'UNITED KINGDOM',
    'UNITED ARAB EMIRATES (UAE)': 'UNITED ARAB EMIRATES',
    'HAWAII': 'USA',
    'CANARY ISLANDS': 'SPAIN',
    'MALAYSIA': 'MALAYSIA',
    'CEYLON (SRI LANKA)': 'SRI LANKA',
    'EQUATORIAL GUINEA / CAMEROON': 'EQUATORIAL GUINEA',  
    'BETWEEN PORTUGAL & INDIA': 'PORTUGAL', 
    'CORAL SEA': 'OTHER',
    'ATLANTIC OCEAN': 'OTHER',
    'CARIBBEAN SEA': 'OTHER',
    'COAST OF AFRICA': 'OTHER',
    'RED SEA?': 'OTHER',
    'ASIA?': 'OTHER',
    'DIEGO GARCIA': 'OTHER', 
    'OKINAWA': 'JAPAN',
    'COLUMBIA': 'COLOMBIA',
    'TRINIDAD': 'TRINIDAD AND TOBAGO',
    'MALDIVE ISLANDS': 'MALDIVES',
    'TURKS': 'TURKS AND CAICOS',
    'REUNION ISLAND': 'OTHER',  

    'CORAL SEA': 'OTHER',
    'ATLANTIC OCEAN': 'OTHER',
    'GULF OF ADEN': 'OTHER',
    'TASMAN SEA': 'OTHER',
    'NORTH ATLANTIC OCEAN': 'OTHER',
    'SOUTH CHINA SEA': 'OTHER',
    'INDEPENDENT STATES': 'OTHER',  

    'RED SEA?' : 'OTHER',
    'ASIA?' : 'OTHER',
    'CEYLON (SRI LANKA)' : 'OTHER',


    # Entradas duplicadas u obsoletas
    'TOBAGO': 'TRINIDAD AND TOBAGO', 
    'BRITISH OVERSEAS TERRITORY': 'OTHER',  
    'PALESTINIAN TERRITORIES': 'OTHER',  
    'ANDAMAN ISLANDS': 'INDIA',  
    'EQUATORIAL GUINEA / CAMEROON': 'EQUATORIAL GUINEA',
    'NEW CALEDONIA': 'FRENCH POLYNESIA',   
    'CEYLON': 'SRI LANKA',
    'NORTHERN ARABIAN SEA': 'OTHER',
    'INDIAN OCEAN?': 'OTHER',
    'AFRICA': 'OTHER',

    'WESTERN SAMOA': 'SAMOA',
    'PACIFIC OCEAN': 'OTHER',
    'BRITISH ISLES': 'UNITED KINGDOM', 
    'NEW BRITAIN': 'PAPUA NEW GUINEA', 
    'JOHNSTON ISLAND': 'OTHER',
    'SOUTH PACIFIC OCEAN': 'OTHER',
    'WEST INDIES': 'OTHER', 
    'BURMA': 'MYANMAR',
    'BRITISH NEW GUINEA': 'PAPUA NEW GUINEA',
    'OCEAN': 'OTHER',
    'MEDITERRANEAN SEA': 'OTHER',
    'ROATAN': 'OTHER',
    'SOUTH CHINA SEA': 'OTHER',
    'KOREA': 'OTHER',
    'MID-PACIFC OCEAN': 'OTHER'
}

In [62]:
### Reemplazamos valores que hacemos 
df_country['country'] = df_country['country'].replace(other)

/var/folders/bk/hzstscb16rn4p883jsg8t31c0000gn/T/ipykernel_4582/2142896039.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_country['country'] = df_country['country'].replace(other)


In [63]:
df_country['country'].count()

np.int64(7014)

In [64]:
### Resolvemos nuestra hipótesis
df_country['country'].value_counts()

country
USA                 2578
AUSTRALIA           1516
SOUTH AFRICA         599
NEW ZEALAND          146
PAPUA NEW GUINEA     143
                    ... 
ARUBA                  1
BAY OF BENGAL          1
SLOVENIA               1
CURACAO                1
BAHREIN                1
Name: count, Length: 161, dtype: int64

-  Descubrimos que los cinco países donde más ataques ha habido son: USA, Australia, South Africa, New Zealand y Papua Nueva Guinea. 

### HIPÓTESIS 5 - 'ACTIVITY': ¿Qué actividad provoca más ataques?

- 'ACTIVITY'

In [65]:
## Repetimos proceso 
df_activity = df[['activity']]

In [66]:
## Vemos los valores que contiene 'activity' y sus cantidades
df_activity['activity'].value_counts()

activity
Surfing                                          1143
Swimming                                         1011
Fishing                                           494
Spearfishing                                      390
Wading                                            178
                                                 ... 
Floating on a small orange raft                     1
Filming & feeding captive sharks                    1
Attempting to drive shark away from the beach       1
Spearfishing / scuba diving                         1
Wreck of  large double sailing canoe                1
Name: count, Length: 1609, dtype: int64

In [67]:
## Lo primero que vamos a hacer, al ser valores tan diferentes, es contar cuátos nulos hay
df_activity['activity'].isnull().sum()

np.int64(585)

In [68]:
## Decidimos eliminar valores nulos 
df_activity = df_activity.dropna(subset=['activity'])

In [69]:
# Para trabajar mejor los pasamos a minúsculas y quitamos espacios para ver si hay coincidencias entre los valores
df_activity['activity'] = df_activity['activity'].str.strip().str.lower()

In [70]:
## Nos damos cuenta de que hay otros valores que sí incluyen las actividades más frecuentes pero que lo ponen en otra categoría porque tiene más texto.
## Para no perder esos valores y sumarlos a nuestro actividades frecuentes primero abrimos una lista con los nombres. 
frequent_activity= [
    'surfing', 'swimming', 'fishing', 'spearfishing', 'wading', 'bathing',
    'diving', 'snorkeling', 'standing', 'scuba diving'
]

In [71]:
## Definimos una función que busque las palabras de tu lista en cada celda. 
def freq_activity(activity):
    for act in frequent_activity:
        if act in activity:
            return activity # Si encuentra la palabra, devuelve esa categoría
            
    return "other" # Si no encuentra ninguna, lo marca como 'other' o el valor original

In [72]:
## Aplicamos la función a nuestro data frame 
df_activity['activity'] = df_activity['activity'].apply(freq_activity)

In [73]:
df_activity.count()

activity    6479
dtype: int64

In [74]:
## Obtenemos el top de actividades más frecuentes
df_activity.value_counts().head(10)

activity    
other           1359
surfing         1148
swimming        1058
fishing          510
spearfishing     398
wading           178
bathing          167
diving           151
snorkeling       135
standing         115
Name: count, dtype: int64

- Del total de actividades registradas (6479), descubrimos que las 10 actividades con más riesgo de ataque son surf, natación, pesca, pesca submarina, pesca con vadeo, bañándose, buceando, haciendo snorkling y estando de pie. Sin embargo, la gran mayoría de personas han sido atacadas por otras actividades tan diferentes que es imposible definir que hay una causa más proclive que otra. 

### HIPÓTESIS 6 - 'DATE': ¿En qué época del año (por trimestres) hay más ataques?

In [75]:
### Hacemos un df más pequeño 
df_date = df[['date']]

In [76]:
### Observamos valores únicos
df_date['date'].unique()

array(['10th January', '8th January', '3rd January ', ..., '1900-1905',
       '1883-1889', '1845-1853'], shape=(6107,), dtype=object)

In [77]:
### Como hay muchos valores muy diferentes pero válidos lo que intentamos es unificarlos 
### Para unificarlos elaboramos una serie de funciones que nos limpien los valores de nuestro df_date

In [78]:
def false_words(s):
    eliminate_words = [
        'Before', 'After', 'No date', 'Late',
        'Circa', 'Ca.', 'Letter dated'
    ]
    
    s = str(s).lower()
    #Recorre la lista eliminate_words comprueba si alguna de esas palabras esta dentro del texto 
    #devuelve True si al menos encuenta una 
    for word in eliminate_words:
        if word.lower() in s:
            return True
    
    return False

In [79]:
def extract_full_month(s):
    #lista de los meses
    months_full = [
        'January', 'February', 'March', 'April', 'May', 'June',
        'July', 'August', 'September', 'October', 'November', 'December'
    ]
    
    #
    months_abbreviate = []
    for m in months_full:
        months_abbreviate.append(m[:3])
    #flags=re.I -> hace la búsqueda a mayúsculas y minúsculas
    #crea pares con el nombre completo del mes y su abreviatura ('January', 'Jan')
    for m_full, m_abbr in zip(months_full, months_abbreviate):
        #primera condición busca el nombre completo del mes y la segunda la abreviatura
        if re.search(rf'\b{m_full}\b', s, flags=re.I) or \
           re.search(rf'\b{m_abbr}\b', s, flags=re.I):
            return m_full
    
    return None

In [80]:
#El trabajo de esta función es limpiar, validar y extraer el mes 
def parse_month(x):
    if pd.isna(x):
        return None
        
    s = str(x).strip()
        
    #Descartar fechas inciertas
    if false_words(s):
        return None
        
    #Extraer mes
    return extract_full_month(s)

In [81]:
### Creamos una columna nueva para aplicar los cambios 
df_date['month_full_strict'] = df_date['date'].apply(parse_month)

/var/folders/bk/hzstscb16rn4p883jsg8t31c0000gn/T/ipykernel_4582/4184569018.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_date['month_full_strict'] = df_date['date'].apply(parse_month)


In [82]:
### Hacemos el cambio en la columna date
df_date['date'] = df_date['date'].apply(parse_month)

/var/folders/bk/hzstscb16rn4p883jsg8t31c0000gn/T/ipykernel_4582/3970362783.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_date['date'] = df_date['date'].apply(parse_month)


In [83]:
#Miramos cuantos None (la nueva categoria que hemos creado para eliminar)
df_date['date'].value_counts(dropna=False)

date
July         775
August       667
None         637
September    608
January      568
June         547
October      502
December     493
April        491
November     457
March        456
May          447
February     416
Name: count, dtype: int64

In [84]:
###.notna() es un metodo que sirve para detectar los valores que no son nulos 
df_date = df_date[df_date['date'].notna()]

In [85]:
### Revisamos que hemos eliminado los None
df_date['date'].value_counts(dropna=False)
df_date['date'].unique()

array(['January', 'December', 'November', 'October', 'September',
       'August', 'July', 'June', 'May', 'February', 'March', 'April'],
      dtype=object)

In [86]:
# Primero aseguramos de que sean strings y sin espacios
df_date['sate'] = df_date['date'].astype(str).str.strip()

In [87]:
trimesters = {
    'January': 'T1 (January,February, March)', 'February': 'T1 (January,February, March)', 'March': 'T1 (January,February, March)',
    'April': 'T2 (April, May, June)', 'May': 'T2 (April, May, June)', 'June': 'T2 (April, May, June)',
    'July': 'T3 (July, August, September)', 'August': 'T3 (July, August, September)', 'September': 'T3 (July, August, September)',
    'October': 'T4 (October, November, December)', 'November': 'T4 (October, November, December)', 'December': 'T4 (October, November, December)'
}

### Aplicamos un MAP para que nos divida los meses por trismestres
df_date['date'] = df_date['date'].map(trimesters)


In [88]:
df_date['date'].count()

np.int64(6427)

In [89]:
### Resolvemos nuestra hipótesis
df_date['date'].value_counts()

date
T3 (July, August, September)        2050
T2 (April, May, June)               1485
T4 (October, November, December)    1452
T1 (January,February, March)        1440
Name: count, dtype: int64

- Para responder a nuestra hipótesis a época del año donde hay más ataques es en el tercer trimestre del año (meses de julio, agosto y septiembre), con 2050 registros. Es decir, un 31,90% del total. 